# Basic Statistics - Descriptive Analytics and Data Preprocessing

This notebook analyzes the Sales and Discounts dataset using descriptive statistics, visualizations, standardization, and one-hot encoding.


## 1. Import Libraries and Load Data


In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, OneHotEncoder

import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='Set2')
pd.set_option('display.max_columns', None)


In [ ]:
df = pd.read_csv('sales_data_with_discounts.csv')
df.head()


In [ ]:
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")
df.info()


## 2. Identify Numerical and Categorical Columns


In [ ]:
numerical_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df.select_dtypes(exclude=np.number).columns.tolist()

print('Numerical columns:')
print(numerical_cols)
print()
print('Categorical columns:')
print(categorical_cols)


In [ ]:
missing_values = df.isnull().sum()
missing_values[missing_values > 0] if missing_values.any() else 'No missing values found.' 


## 3. Descriptive Analytics for Numerical Columns

The table below includes the required mean, median, mode, and standard deviation for each numerical column.


In [ ]:
descriptive_stats = pd.DataFrame({
    'Mean': df[numerical_cols].mean(),
    'Median': df[numerical_cols].median(),
    'Mode': df[numerical_cols].mode().iloc[0],
    'Standard Deviation': df[numerical_cols].std(),
    'Skewness': df[numerical_cols].skew(),
    'Minimum': df[numerical_cols].min(),
    'Maximum': df[numerical_cols].max()
})

descriptive_stats.round(2)


In [ ]:
df[numerical_cols].describe().round(2)


### Interpretation of Descriptive Statistics

- **Mean:** The average values show that the typical transaction has around 5 units in `Volume`, but the average `Net Sales Value` is much higher than the median. This means a few expensive or high-value transactions increase the overall average.
- **Median:** The median is lower than the mean for `Avg Price`, `Total Sales Value`, `Discount Amount`, and `Net Sales Value`. This confirms that most transactions are smaller, while a smaller number of high-value transactions pull the mean upward.
- **Mode:** The mode shows the most frequently occurring value. For variables like `Volume`, the common purchase quantity is low, which suggests customers usually buy fewer units per transaction.
- **Standard Deviation:** Sales-related columns have high standard deviation, meaning revenue varies widely across products. This is expected because the dataset includes low-price FMCG items as well as high-price mobile products.
- **Skewness:** `Volume`, `Avg Price`, `Total Sales Value`, `Discount Amount`, and `Net Sales Value` are positively skewed. In business terms, a small number of premium products or large transactions contribute a large share of revenue.
- **Discount Rate (%):** This column is negatively skewed, meaning many discounts are concentrated toward the higher end of the discount range.


## 4. Histograms for Numerical Columns

Each histogram shows the distribution of one numerical variable.


In [ ]:
n_cols = 2
n_rows = math.ceil(len(numerical_cols) / n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(13, 4 * n_rows))
axes = axes.flatten()

for ax, col in zip(axes, numerical_cols):
    sns.histplot(df[col], bins=30, kde=True, ax=ax, color='steelblue')
    ax.set_title(f'Distribution of {col}')
    ax.set_xlabel(col)
    ax.set_ylabel('Frequency')

for ax in axes[len(numerical_cols):]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()


### Histogram Inference

- `Volume` is right-skewed, so most orders contain a small number of units, while only a few orders have very high quantities.
- `Avg Price` is strongly right-skewed because the dataset contains many low-priced FMCG/lifestyle products and fewer high-priced mobile products.
- `Total Sales Value` and `Net Sales Value` are right-skewed, meaning revenue is not evenly distributed across transactions. A smaller group of transactions contributes disproportionately to total revenue.
- `Discount Amount` is also right-skewed because high-value products receive larger absolute discount amounts.
- `Discount Rate (%)` is distributed differently from sales values; discount percentages are bounded between about 5% and 20%, so the spread is much narrower.

**Business insight:** The company should pay special attention to high-value transactions because they have a large effect on revenue. However, the large number of smaller transactions still matters for volume and customer reach.


## 5. Boxplots for Numerical Columns

Boxplots help identify the interquartile range and outliers.


In [ ]:
n_cols = 2
n_rows = math.ceil(len(numerical_cols) / n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(13, 3.5 * n_rows))
axes = axes.flatten()

for ax, col in zip(axes, numerical_cols):
    sns.boxplot(x=df[col], ax=ax, color='lightcoral')
    ax.set_title(f'Boxplot of {col}')
    ax.set_xlabel(col)

for ax in axes[len(numerical_cols):]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()


In [ ]:
outlier_summary = []

for col in numerical_cols:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    outlier_summary.append({
        'Column': col,
        'Q1': q1,
        'Q3': q3,
        'IQR': iqr,
        'Lower Bound': lower_bound,
        'Upper Bound': upper_bound,
        'Outlier Count': len(outliers)
    })

pd.DataFrame(outlier_summary).round(2)


### Boxplot Inference

- Outliers are visible in `Volume`, `Avg Price`, `Total Sales Value`, `Discount Amount`, and `Net Sales Value`.
- These outliers should not be removed automatically because they may represent real business events, such as premium mobile sales or large purchases.
- `Avg Price` outliers show that product price levels differ greatly across business units.
- `Discount Amount` outliers are linked to expensive products: even a normal discount rate produces a large discount amount when the base price is high.
- `Net Sales Value` outliers indicate transactions that generate unusually high revenue after discounts.

**Business insight:** Outliers here are commercially important. Instead of treating them only as statistical noise, management can study them to understand which products and categories drive the highest sales value.


## 6. Bar Chart Analysis for Categorical Columns

Bar charts below show the count/frequency of each category.


In [ ]:
plot_categorical_cols = categorical_cols.copy()
n_cols = 2
n_rows = math.ceil(len(plot_categorical_cols) / n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
axes = axes.flatten()

for ax, col in zip(axes, plot_categorical_cols):
    order = df[col].value_counts().index
    sns.countplot(data=df, x=col, order=order, ax=ax, color='mediumseagreen')
    ax.set_title(f'Frequency of {col}')
    ax.set_xlabel(col)
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=45)

for ax in axes[len(plot_categorical_cols):]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()


In [ ]:
category_summary = {
    col: df[col].value_counts().head(10)
    for col in categorical_cols
}

for col, counts in category_summary.items():
    print()
    print(f'{col}:')
    print(counts)


### Categorical Variable Inference

- `City` has only one value, so it does not help explain differences in sales behavior.
- `BU` is perfectly balanced with 150 records each for Mobiles, FMCG, and Lifestyle. This makes comparison between business units fair.
- `SKU` and `Model` have many categories, with each appearing repeatedly. These columns are useful for product-level analysis but increase the number of one-hot encoded columns.
- `Brand` distribution is uneven. For example, some brands appear more frequently than others, which may affect total brand-level sales.
- `Date` is repeated across multiple products each day. It is useful for time-based analysis, but for basic preprocessing it can create many dummy columns.

**Business insight:** Product category and brand are more useful than city in this dataset. Since city does not vary, the main sales differences come from product type, price, brand, model, discount, and volume.


## 7. Relationship Between Sales Variables


In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x='Total Sales Value', y='Net Sales Value', hue='BU')
plt.title('Total Sales Value vs Net Sales Value')
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df[numerical_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap for Numerical Columns')
plt.tight_layout()
plt.show()


In [ ]:
business_summary = df.groupby('BU').agg(
    Average_Volume=('Volume', 'mean'),
    Average_Price=('Avg Price', 'mean'),
    Total_Sales=('Total Sales Value', 'sum'),
    Total_Discount=('Discount Amount', 'sum'),
    Net_Sales=('Net Sales Value', 'sum')
).round(2)

business_summary


### Relationship and Business Inference

- `Total Sales Value` and `Net Sales Value` have a very strong positive relationship because net sales are calculated after subtracting discounts from total sales.
- `Discount Rate (%)` has a strong negative relationship with `Net Sales Value`. Higher discount percentages reduce the final revenue collected from each transaction.
- `Avg Price` has a strong positive relationship with sales-value columns because expensive products naturally produce higher revenue.
- `Volume` is not the only driver of revenue. A low-volume mobile transaction can produce more sales value than a high-volume FMCG transaction because price differs greatly.
- Mobiles generate the highest net sales because their average price is much higher, even though average volume is lower than FMCG.

**Business insight:** Discount strategy should be monitored carefully. Discounts can increase customer attraction, but high discount rates reduce net sales. The business should compare whether discounted products create enough additional volume to justify the revenue reduction.


## 8. Standardization of Numerical Variables

Standardization converts numerical columns to z-scores using:

`z = (x - mean) / standard deviation`

After standardization, each numerical column has approximately mean 0 and standard deviation 1.


In [ ]:
scaler = StandardScaler()
standardized_array = scaler.fit_transform(df[numerical_cols])
df_standardized = pd.DataFrame(standardized_array, columns=numerical_cols)

df_standardized.head()


In [ ]:
comparison = pd.DataFrame({
    'Original Mean': df[numerical_cols].mean(),
    'Original Std': df[numerical_cols].std(),
    'Standardized Mean': df_standardized.mean(),
    'Standardized Std': df_standardized.std()
})

comparison.round(3)


In [ ]:
fig, axes = plt.subplots(len(numerical_cols), 2, figsize=(12, 22))

for row, col in enumerate(numerical_cols):
    sns.histplot(df[col], bins=30, kde=True, ax=axes[row, 0], color='steelblue')
    axes[row, 0].set_title(f'Original {col}')
    axes[row, 0].set_xlabel(col)

    sns.histplot(df_standardized[col], bins=30, kde=True, ax=axes[row, 1], color='darkorange')
    axes[row, 1].set_title(f'Standardized {col}')
    axes[row, 1].set_xlabel(f'Standardized {col}')

plt.tight_layout()
plt.show()


### Standardization Inference

- Before standardization, variables such as `Avg Price`, `Total Sales Value`, and `Net Sales Value` dominate because their numerical scale is much larger than `Volume` or `Discount Rate (%)`.
- After standardization, each numerical column is converted to a comparable scale with mean close to 0 and standard deviation close to 1.
- The distribution shape does not change after standardization. Skewness and outliers remain visible because standardization changes scale, not the underlying pattern.
- This step is important for algorithms such as KNN, clustering, PCA, logistic regression, and neural networks because those methods are affected by feature magnitude.

**Business insight:** Standardization helps models treat volume, price, discount, and sales value fairly instead of allowing high-value currency columns to dominate the analysis.


## 9. Convert Categorical Data into Dummy Variables

One-hot encoding converts categorical values into binary columns so machine learning algorithms can use them.


In [ ]:
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoded_array = encoder.fit_transform(df[categorical_cols])
encoded_columns = encoder.get_feature_names_out(categorical_cols)

df_encoded = pd.DataFrame(encoded_array, columns=encoded_columns, index=df.index)

encoded_feature_count = pd.DataFrame({
    'Categorical Column': categorical_cols,
    'Unique Categories': [df[col].nunique() for col in categorical_cols]
})
encoded_feature_count['Dummy Columns Created'] = encoded_feature_count['Unique Categories']

print(f'Total dummy columns created: {df_encoded.shape[1]}')
encoded_feature_count


In [ ]:
processed_df = pd.concat([df_standardized, df_encoded], axis=1)

print(f'Original dataset shape: {df.shape}')
print(f'Standardized numerical columns: {df_standardized.shape[1]}')
print(f'One-hot encoded categorical columns: {df_encoded.shape[1]}')
print(f'Processed dataset shape: {processed_df.shape}')

processed_df.head()


### One-Hot Encoding Inference

- One-hot encoding created one binary column for each unique category in the categorical variables.
- The processed dataset has 101 columns: 6 standardized numerical columns and 95 dummy-variable columns.
- Columns such as `SKU`, `Model`, and `Date` create many dummy variables because they contain many unique values.
- `City` creates only one dummy column because all records belong to the same city.
- `handle_unknown='ignore'` is useful because it prevents errors if new category values appear later during model prediction.

**Business insight:** One-hot encoding allows models to learn differences between brands, product models, business units, and days. However, high-cardinality variables increase dataset size, so they should be handled carefully in larger real-world datasets.


## 10. Conclusion

This analysis shows that the dataset contains 450 records, 6 numerical columns, and 7 categorical columns. The most important pattern is that sales values are strongly right-skewed: most transactions are relatively small, while a smaller number of premium or high-value transactions generate a large share of revenue.

The descriptive statistics show that the mean is much higher than the median for `Avg Price`, `Total Sales Value`, `Discount Amount`, and `Net Sales Value`. This confirms the presence of high-value observations. The boxplots also show outliers, but these outliers are likely meaningful business transactions rather than data errors.

The visualizations show that mobiles contribute high sales value because of high average price, while FMCG has lower average price but higher average volume. Discount rate is negatively related to net sales, so discount decisions should be evaluated carefully. A higher discount may support sales volume, but it directly reduces revenue per transaction.

Standardization prepared the numerical variables by putting them on a common scale. One-hot encoding transformed categorical variables into machine-learning-ready binary columns. Together, these preprocessing steps make the dataset more suitable for further analysis and predictive modeling.

**Final business takeaway:** Revenue is mainly driven by product price, business unit, brand/model, and discount strategy. The company should monitor high-value product segments and evaluate whether discount campaigns are increasing volume enough to compensate for lower net sales.
